In [2]:
import pandas as pd

# Load the training data from train.parquet
df = pd.read_parquet("train.parquet")

# Print out the columns to see what's available
print("Columns in training data:", df.columns.tolist())

# Display the first few rows to inspect the data
print(df.head())

Columns in training data: ['id', 'content', 'lang', 'manipulative', 'techniques', 'trigger_words']
                                     id  \
0  0bb0c7fa-101b-4583-a5f9-9d503339141c   
1  7159f802-6f99-4e9d-97bd-6f565a4a0fae   
2  e6a427f1-211f-405f-bd8b-70798458d656   
3  1647a352-4cd3-40f6-bfa1-d87d42e34eea   
4  9c01de00-841f-4b50-9407-104e9ffb03bf   

                                             content lang  manipulative  \
0  Новий огляд мапи DeepState від російського вій...   uk          True   
1  Недавно 95 квартал жёстко поглумился над русск...   ru          True   
2  🤩\nТим часом йде евакуація Бєлгородського авто...   uk          True   
3  В Україні найближчим часом мають намір посилит...   uk         False   
4  Расчёты 122-мм САУ 2С1 "Гвоздика" 132-й бригад...   ru          True   

                          techniques  \
0        [euphoria, loaded_language]   
1  [loaded_language, cherry_picking]   
2        [loaded_language, euphoria]   
3                              

In [3]:
import numpy as np
import ast

def parse_trigger_words(row):
    # If row is already a list, tuple, or NumPy array, convert to list
    if isinstance(row, (list, tuple, np.ndarray)):
        return list(row)
    # If row is None, NaN, or literally "None", return empty list
    if row is None or pd.isna(row) or str(row).strip() == "None":
        return []
    try:
        # Attempt to parse a string like "[[27, 63], [65, 88]]" -> [[27, 63], [65, 88]]
        return ast.literal_eval(str(row).strip())
    except Exception as e:
        print(f"Error parsing trigger_words: {row} -> {e}")
        return []

# Apply the parsing function to create a new column "trigger_spans"
df["trigger_spans"] = df["trigger_words"].apply(parse_trigger_words)

# Inspect the first 5 rows to confirm the new column
df[["trigger_words", "trigger_spans"]].head()

,trigger_words,trigger_spans
0,"[[27, 63], [65, 88], [90, 183], [186, 308]]","[[27, 63], [65, 88], [90, 183], [186, 308]]"
1,"[[0, 40], [123, 137], [180, 251], [253, 274]]","[[0, 40], [123, 137], [180, 251], [253, 274]]"
2,"[[55, 100]]","[[55, 100]]"
3,None,[]
4,"[[114, 144]]","[[114, 144]]"


In [4]:
df.rename(columns={"content": "text"}, inplace=True)

# Let's confirm
df[["text", "trigger_spans"]].head()

,text,trigger_spans
0,Новий огляд мапи DeepState від російського вій...,"[[27, 63], [65, 88], [90, 183], [186, 308]]"
1,Недавно 95 квартал жёстко поглумился над русск...,"[[0, 40], [123, 137], [180, 251], [253, 274]]"
2,🤩\nТим часом йде евакуація Бєлгородського авто...,"[[55, 100]]"
3,В Україні найближчим часом мають намір посилит...,[]
4,"Расчёты 122-мм САУ 2С1 ""Гвоздика"" 132-й бригад...","[[114, 144]]"


In [5]:
def create_input_target(row):
    # Convert each array-like span to a standard Python tuple of Python ints
    # so your final string is purely numeric (e.g. (27,63)) rather than (np.int64(27), np.int64(63))
    spans_as_tuples = [(int(span[0]), int(span[1])) for span in row["trigger_spans"]]

    # Construct the input text with a simple prompt
    input_text = "Extract manipulative spans: " + row["text"]

    # Convert the Python list of tuples into a string, e.g. "[(27, 63), (65, 88)]"
    target_text = str(spans_as_tuples)

    return {"input_text": input_text, "target_text": target_text}

df_seq2seq = df.apply(create_input_target, axis=1, result_type="expand")
df_seq2seq.head()

,input_text,target_text
0,Extract manipulative spans: Новий огляд мапи D...,"[(27, 63), (65, 88), (90, 183), (186, 308)]"
1,Extract manipulative spans: Недавно 95 квартал...,"[(0, 40), (123, 137), (180, 251), (253, 274)]"
2,Extract manipulative spans: 🤩\nТим часом йде е...,"[(55, 100)]"
3,Extract manipulative spans: В Україні найближч...,[]
4,Extract manipulative spans: Расчёты 122-мм САУ...,"[(114, 144)]"


In [6]:
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 18.2 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system 

In [7]:
from datasets import Dataset

# Convert your DataFrame to a Hugging Face Dataset
seq2seq_dataset = Dataset.from_pandas(df_seq2seq)

# Inspect one example to ensure we have 'input_text' and 'target_text'
print(seq2seq_dataset[0])

{'input_text': 'Extract manipulative spans: Новий огляд мапи DeepState від російського військового експерта, кухара путіна 2 розряду, спеціаліста по снарядному голоду та ректора музичної академії міноборони рф Євгєнія Пригожина. \nПригожин прогнозує, що невдовзі настане день звільнення Криму і день розпаду росії. Каже, що передумови цього вже створені. \n*Відео взяли з каналу \nФД\n. \n@informnapalm', 'target_text': '[(27, 63), (65, 88), (90, 183), (186, 308)]'}


In [8]:
split_dataset = seq2seq_dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

print("Train size:", len(train_dataset))
print("Eval size:", len(eval_dataset))

Train size: 3057
Eval size: 765


In [9]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/mt5-base"

tokenizer_seq2seq = AutoTokenizer.from_pretrained(model_name, use_fast=True)
model_seq2seq = AutoModelForSeq2SeqLM.from_pretrained(model_name)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/376 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/702 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
/usr/local/lib/python3.11/dist-packages/transformers/convert_slow_tokenizer.py:559: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/2.33G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.33G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [10]:
def preprocess_function(examples):
    # Tokenize the input prompt + text
    model_inputs = tokenizer_seq2seq(
        examples["input_text"],
        max_length=256,
        truncation=True,
        padding="max_length"
    )

    # Tokenize the target: This is the string of span tuples (e.g., "[(27, 63), (65, 88)]")
    # We switch the tokenizer to "target" mode
    with tokenizer_seq2seq.as_target_tokenizer():
        labels = tokenizer_seq2seq(
            examples["target_text"],
            max_length=64,
            truncation=True,
            padding="max_length"
        )
    # Attach labels
    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

In [11]:
tokenized_train_dataset = train_dataset.map(preprocess_function, batched=True)
tokenized_eval_dataset = eval_dataset.map(preprocess_function, batched=True)

Map:   0%|          | 0/3057 [00:00<?, ? examples/s]

/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:3980: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


Map:   0%|          | 0/765 [00:00<?, ? examples/s]

In [12]:
print(tokenized_train_dataset[0].keys())
print("Example input_ids:", tokenized_train_dataset[0]["input_ids"])
print("Example labels:", tokenized_train_dataset[0]["labels"])

dict_keys(['input_text', 'target_text', 'input_ids', 'attention_mask', 'labels'])
Example input_ids: [67893, 49448, 12998, 14156, 263, 267, 6649, 182202, 259, 80229, 411, 55852, 3671, 259, 5436, 91926, 14078, 259, 264, 259, 105868, 9268, 3152, 259, 4981, 14789, 129189, 51948, 259, 264, 1436, 7357, 1633, 259, 21688, 1008, 259, 70422, 260, 867, 3349, 48526, 9440, 259, 3636, 11456, 56526, 172538, 892, 3800, 182202, 259, 3331, 261, 2531, 259, 113713, 1140, 259, 83546, 543, 1066, 13703, 15351, 15067, 396, 315, 41960, 324, 374, 7357, 1633, 6660, 26166, 433, 260, 313, 152863, 3229, 311, 12977, 3695, 425, 2693, 66600, 15912, 259, 279, 90016, 1915, 106085, 170274, 260, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

In [13]:
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="./results-mt5-span",  # Where to save checkpoints & final model
    evaluation_strategy="steps",       # Evaluate periodically rather than at end of epoch only
    eval_steps=300,                    # Evaluate every 500 training steps
    save_steps=300,                    # Save a checkpoint every 500 steps
    num_train_epochs=3,                # Number of full passes through the dataset
    per_device_train_batch_size=4,     # Batch size per GPU for training
    per_device_eval_batch_size=4,      # Batch size per GPU for evaluation
    learning_rate=2e-5,                # Initial learning rate
    weight_decay=0.01,                 # Strength of L2 regularization
    logging_steps=100,                 # Log training loss & metrics every 100 steps
    save_total_limit=2,                # Keep only the 2 most recent checkpoints
    fp16=True,                         # Mixed-precision training for speed on T4 GPU
    predict_with_generate=True,        # Allows generating text outputs during evaluation
    report_to="none",                  # Disable logging integrations like WandB or TensorBoard
)

/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [14]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    model=model_seq2seq,
    args=training_args,              # the Seq2SeqTrainingArguments you defined
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_eval_dataset,
    tokenizer=tokenizer_seq2seq
)

<ipython-input-14-9bf567252d51>:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


In [15]:
trainer.train()

Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Step,Training Loss,Validation Loss
500,0.000000,nan
1000,0.000000,nan
1500,0.000000,nan
2000,0.000000,nan


TrainOutput(global_step=2295, training_loss=0.0, metrics={'train_runtime': 847.7088, 'train_samples_per_second': 10.819, 'train_steps_per_second': 2.707, 'total_flos': 5498226036965376.0, 'train_loss': 0.0, 'epoch': 3.0})

In [16]:
trainer.save_model("./mt5_span_extraction")
tokenizer_seq2seq.save_pretrained("./mt5_span_extraction")

('./mt5_span_extraction/tokenizer_config.json',
 './mt5_span_extraction/special_tokens_map.json',
 './mt5_span_extraction/spiece.model',
 './mt5_span_extraction/added_tokens.json',
 './mt5_span_extraction/tokenizer.json')

In [17]:
eval_results = trainer.evaluate()
print("Evaluation results:", eval_results)

Evaluation results: {'eval_loss': nan, 'eval_runtime': 12.2267, 'eval_samples_per_second': 62.568, 'eval_steps_per_second': 15.703, 'epoch': 3.0}


In [18]:
trainer.save_model("./mt5_span_extraction")
tokenizer_seq2seq.save_pretrained("./mt5_span_extraction")

('./mt5_span_extraction/tokenizer_config.json',
 './mt5_span_extraction/special_tokens_map.json',
 './mt5_span_extraction/spiece.model',
 './mt5_span_extraction/added_tokens.json',
 './mt5_span_extraction/tokenizer.json')

In [19]:
import pandas as pd

# 1. Load the test.csv file
import csv

df_test = pd.read_csv(
    "test.csv",
    sep=",",                # or "\t" if tab-separated
    quoting=csv.QUOTE_NONE, # avoid handling quotes
    engine="python",
    on_bad_lines="skip"
)


# 2. Check the columns and a few samples
print("Columns in test data:", df_test.columns.tolist())
print(df_test.head(3))

Columns in test data: ['id', 'content']
                                                                                                                                                                                                                              id  \
521cd2e8-dd9f-42c4-98ba-c0c8890ff1ba               "Они просрали нашу технику  положили кучу людей  выставили НАТО дебилами  которые не способны спланировать наступательну...  внесли раскол в наши парламенты  срочно   срочно   
9b2a61e4-d14e-4ff7-b304-e73d720319bf               "❗️                        NaN                  NaN                      NaN                                                NaN                              NaN         None   
Китай предлагает отдать оккупированные территор... NaN                        NaN                  NaN                      NaN                                                NaN                              NaN         None   

                                               

In [20]:
# 1. Rename the text column if your file calls it "content".
#    Only run this if the DataFrame indeed has a column named "content".
if "content" in df_test.columns:
    df_test.rename(columns={"content": "text"}, inplace=True)

# 2. Create the 'input_text' column with the prompt
df_test["input_text"] = "Extract manipulative spans: " + df_test["text"]

# Check the first few rows to confirm changes
print(df_test.head(3))

                                                                                                                                                                                                                              id  \
521cd2e8-dd9f-42c4-98ba-c0c8890ff1ba               "Они просрали нашу технику  положили кучу людей  выставили НАТО дебилами  которые не способны спланировать наступательну...  внесли раскол в наши парламенты  срочно   срочно   
9b2a61e4-d14e-4ff7-b304-e73d720319bf               "❗️                        NaN                  NaN                      NaN                                                NaN                              NaN         None   
Китай предлагает отдать оккупированные территор... NaN                        NaN                  NaN                      NaN                                                NaN                              NaN         None   

                                                                                       

In [21]:
from datasets import Dataset

# We'll keep only the columns we need: "id" and "input_text"
test_dataset = Dataset.from_pandas(df_test[["id", "input_text"]])

# Inspect the first example to confirm "id" and "input_text" are present
print("Test dataset preview:", test_dataset[0])

Test dataset preview: {'id': ' срочно', 'input_text': 'Extract manipulative spans:  давайте им дадим еще триллиард а иначе усэ. Вот так это прямо и работает."', '__index_level_0__': '521cd2e8-dd9f-42c4-98ba-c0c8890ff1ba', '__index_level_1__': '"Они просрали нашу технику', '__index_level_2__': ' положили кучу людей', '__index_level_3__': ' выставили НАТО дебилами', '__index_level_4__': ' которые не способны спланировать наступательную операцию', '__index_level_5__': ' внесли раскол в наши парламенты', '__index_level_6__': ' срочно'}
